We Load the package and function for analyse the results

In [1]:
%load_ext autoreload
%autoreload 2
import pandas as pd
from pathlib import Path
import sys
sys.path.append("..")
from config import models
from src.volatility import target_volatility
from src.plot import volatility_comparison, time_error, violation_plot

Load the result calculated in main.py

In [2]:
BASE_DIR = Path().resolve().parent
stats_path = BASE_DIR / "results"/ "stats"
stats_var_path = BASE_DIR / "results"/ "var"
violation_path = BASE_DIR / "results"/ "violation rate"
test_kupiec_path = BASE_DIR / "results"/ "kupiec"
hits_path = BASE_DIR / "results" / "hits"
forecast_path = BASE_DIR / "results" / "forecast"
forecast_path.mkdir(parents = True, exist_ok = True)
returns_path = BASE_DIR / "results" / "returns"
returns_path.mkdir(parents = True, exist_ok = True)

stats = pd.read_parquet(stats_path / f"metriche_modelli.parquet")
violation_stats = pd.read_parquet(violation_path / f"violation.parquet")
kupiec = pd.read_parquet(test_kupiec_path / f"kupiec.parquet")
var = {}
for model in models:
    var[model] = pd.read_parquet(stats_var_path / f"var{model}.parquet")
hits = {}
for model in models:
    hits[model] = pd.read_parquet(hits_path / f"hits{model}.parquet")
forecast_volatility = {}
for model in models:
    forecast_volatility[model] = pd.read_parquet(forecast_path / f"forecast{model}.parquet")
log_returns = pd.read_parquet(returns_path / f"returns.parquet")

Compute error and add a column in VaR for become the analysis more complete

In [3]:
realized_volatility = target_volatility(log_returns).dropna()
common_index = realized_volatility.index.intersection(forecast_volatility["garch"].index)
realized_volatility = realized_volatility.loc[common_index]
forecast_volatility_aligned = {
    model: forecast_volatility[model].loc[common_index] for model in models
}
error = {}
for model in models:
    error[model] = (realized_volatility - forecast_volatility_aligned[model])

Var = []
for model in models:
    vr = var[model].copy()
    vr["model"] = model
    Var.append(vr)
VaR = pd.concat(Var, ignore_index = True)

Create plots for an easy and immediate visualization of main results

In [4]:
volatility_comparison(forecast_volatility_aligned, "AAPL", realized_volatility)
time_error(error, "AAPL")
violation_plot(hits, "AAPL")

print the main result for evaluate the different models. metrics, violations and kupiec test

In [5]:
print(stats.groupby("model").mean())
print(violation_stats.groupby("model").mean())
print(VaR.groupby("model").mean())
print(kupiec["reject p-value"].groupby("model").mean())
print(kupiec["reject p-value"].groupby("model").describe())

              MAE      RMSE    Q-LIKE
model                                
ewma     0.002144  0.003194 -7.319890
garch    0.003319  0.005274 -7.261694
rolling  0.004196  0.006264 -7.220629
              T  violation rate  violation
model                                     
ewma     2452.0        0.063723     156.25
garch    2452.0        0.061399     150.55
rolling  2452.0        0.060644     148.70
             AAPL      AMZN       AXP        BA       CAT       CVX        DE  \
model                                                                           
ewma    -0.025623 -0.029732 -0.019796 -0.032062 -0.029166 -0.021566 -0.027307   
garch   -0.026004 -0.030312 -0.020190 -0.032316 -0.030156 -0.021750 -0.028083   
rolling -0.025919 -0.030090 -0.020194 -0.032546 -0.029436 -0.021916 -0.027694   

              DIS         F       FDX     GOOGL        GS       JNJ        KO  \
model                                                                           
ewma    -0.022925 -0.032595

ewma is the best model if we compare the error's functions, garch is the second best and rolling is the worst.

While if we compare the violations rate the results change. rolling become the less miscalibrated and ewma the worst.
infact by the kupiec test rolling has 9 rejection while ewma and garch 12.

Nevertheless, rolling remain the worst for forecast accuracy.